In [1]:
import pandas as pd
import duckdb

df_device = pd.DataFrame({
    "device_id": [
        "R05", "R16", "R34", "R36"
    ],
    "device_name": [
        "前向散射仪",
        "云高仪",
        "跑道视程仪",
        "自动气象站"
    ],
    "site": [
        "05端",
        "16端",
        "34端",
        "36端"
    ]
})


df_alarm = pd.DataFrame({
    "alarm_id": [
        101, 102, 103, 104, 105, 106, 107
    ],
    "device_id": [
        "R05", "R05", "R16",
        "R34", "R34", "R34",
        "R36"
    ],
    "alarm_level": [
        "ERROR",
        "WARNING",
        "ERROR",
        "WARNING",
        "ERROR",
        "ERROR",
        "INFO"
    ],
    "alarm_time": [
        "2026-07-29 08:00:00",
        "2026-07-29 09:00:00",
        "2026-07-29 10:00:00",
        "2026-07-29 07:00:00",
        "2026-07-29 08:00:00",
        "2026-07-29 09:00:00",
        "2026-07-29 11:00:00"
    ]
})


df_maintenance = pd.DataFrame({
    "order_id": [
        201, 202, 203, 204
    ],
    "device_id": [
        "R05", "R16", "R34", "R34"
    ],
    "order_status": [
        "COMPLETED",
        "OPEN",
        "COMPLETED",
        "COMPLETED"
    ]
})


df_alarm["alarm_time"] = pd.to_datetime(
    df_alarm["alarm_time"]
)

df_device

,device_id,device_name,site
0,R05,前向散射仪,05端
1,R16,云高仪,16端
2,R34,跑道视程仪,34端
3,R36,自动气象站,36端


In [2]:
df_alarm

,alarm_id,device_id,alarm_level,alarm_time
0,101,R05,ERROR,2026-07-29 08:00:00
1,102,R05,WARNING,2026-07-29 09:00:00
2,103,R16,ERROR,2026-07-29 10:00:00
3,104,R34,WARNING,2026-07-29 07:00:00
4,105,R34,ERROR,2026-07-29 08:00:00
5,106,R34,ERROR,2026-07-29 09:00:00
6,107,R36,INFO,2026-07-29 11:00:00


In [3]:
df_maintenance 

,order_id,device_id,order_status
0,201,R05,COMPLETED
1,202,R16,OPEN
2,203,R34,COMPLETED
3,204,R34,COMPLETED


# SQL Daily Review：设备告警与维修统计分析

## 题目背景

现在需要对设备运行情况进行统计分析。

已有三张表：

- 设备基础信息表 `df_device`
- 告警记录表 `df_alarm`
- 维修工单表 `df_maintenance`

需要统计每台设备的：

1. ERROR 告警次数；
2. 已完成维修工单数量。

即使设备没有告警或维修记录，也需要保留设备信息。

---

## 题目要求

以设备表：

```text
df_device
```

作为主表。

关联：

```text
df_alarm
df_maintenance
```

统计：

### ERROR 告警次数

只统计：

```text
alarm_level = 'ERROR'
```

### 已完成维修数量

只统计：

```text
order_status = 'COMPLETED'
```

---

## 输出字段

| 字段 | 含义 |
|---|---|
| device_id | 设备编号 |
| device_name | 设备名称 |
| site | 安装位置 |
| error_alarm_count | ERROR告警次数 |
| completed_order_count | 已完成维修数量 |

---

## 注意事项

### 1. 没有记录的设备也要保留

例如：

```text
R36
```

没有 ERROR 告警，也没有完成维修。

结果应该显示：

```text
error_alarm_count = 0
completed_order_count = 0
```

---

### 2. 不允许因为 JOIN 导致重复计数

例如：

设备 R34：

```text
ERROR告警 2条
维修工单 2条
```

如果直接两个表 JOIN：

可能产生：

```text
2 × 2 = 4 行
```

导致数量错误。

需要提前考虑如何避免重复统计。

---

## 最终排序

按照：

1. device_id 升序

---

## 解题要求

- 使用 JOIN；
- 使用 LEFT JOIN 保留所有设备；
- 使用聚合统计；
- 不使用子查询套子查询；
- 注意多表 JOIN 后的重复计数问题。

In [20]:
query = """
WITH alarm_count AS (
    SELECT
        device_id,
        SUM(
            CASE
                WHEN alarm_level = 'ERROR' THEN 1
                ELSE 0
            END
        )::INTEGER AS error_alarm_count
    FROM df_alarm
    GROUP BY device_id
),

maintenance_count AS (
    SELECT
        device_id,
        SUM(
            CASE
                WHEN order_status = 'COMPLETED' THEN 1
                ELSE 0
            END
        )::INTEGER AS completed_order_count
    FROM df_maintenance
    GROUP BY device_id
)

SELECT
    dd.device_id,
    dd.device_name,
    dd.site,
    COALESCE(ac.error_alarm_count, 0) AS error_alarm_count,
    COALESCE(mc.completed_order_count, 0) AS completed_order_count
FROM df_device AS dd
LEFT JOIN alarm_count AS ac
    ON dd.device_id = ac.device_id
LEFT JOIN maintenance_count AS mc
    ON dd.device_id = mc.device_id
ORDER BY dd.device_id;
"""

df = duckdb.execute(query).fetchdf()
df

,device_id,device_name,site,error_alarm_count,completed_order_count
0,R05,前向散射仪,05端,1,1
1,R16,云高仪,16端,1,0
2,R34,跑道视程仪,34端,2,2
3,R36,自动气象站,36端,0,0
